# IMS Chapter 8: Inference for Regression

Python code examples for **Chapter 8** of *Introduction to Modern Statistics*, covering:

1. **SLR inference** — bootstrap and t-test for the slope
2. **Model assumptions** — L-I-N-E diagnostics
3. **MLR inference** — multiple coefficients and their p-values
4. **Logistic regression** — log-odds, odds ratios, predicted probabilities

**Data files needed**: Place `loans_full_schema.csv` and `email.csv` in this folder before running Sections 8.3 and 8.4.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
import warnings
warnings.filterwarnings('ignore')

np.random.seed(47)
plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
print('Libraries loaded.')

---
---
# Chapter 8: Inference for Regression

This chapter applies inferential ideas to regression models:
1. **SLR inference** — bootstrap and t-test for slope
2. **Model assumptions** — L-I-N-E diagnostics
3. **MLR inference** — interpreting coefficients and p-values
4. **Logistic regression inference** — coefficients as log-odds

---
## 8.1 Inference for Simple Linear Regression

### The population model

$$y = \beta_0 + \beta_1 x + \varepsilon, \quad \varepsilon \sim N(0, \sigma^2)$$

We estimate the slope $\beta_1$ with the least-squares estimate $b_1$.

### Example: Sandwich store advertising vs. revenue

A chain sandwich restaurant believes monthly revenue ($K) is a linear function of advertising spend ($K). We simulate the *true* population (which the CEO knows) and take a sample of 20 stores.

In [ ]:
# True population parameters
beta0_true = 11.23
beta1_true = 4.80

rng11 = np.random.default_rng(4747)
pop_size = 1000
ad_pop  = rng11.normal(4, 1, pop_size)
rev_pop = beta0_true + beta1_true * ad_pop + rng11.normal(0, 8, pop_size)

# Take ONE sample of 20 stores
idx = rng11.choice(pop_size, size=20, replace=False)
ad_samp  = ad_pop[idx]
rev_samp = rev_pop[idx]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.scatter(ad_pop, rev_pop, alpha=0.3, s=10, color='grey', label='Population')
m_pop, b_pop = np.polyfit(ad_pop, rev_pop, 1)
x_line = np.linspace(ad_pop.min(), ad_pop.max(), 100)
ax.plot(x_line, m_pop*x_line + b_pop, color='steelblue', linewidth=2)
ax.set_xlabel('Advertising spend ($K)')
ax.set_ylabel('Revenue ($K)')
ax.set_title(f'Full population (truth: β₀={beta0_true}, β₁={beta1_true})')

ax2 = axes[1]
ax2.scatter(ad_samp, rev_samp, color='steelblue', label='Sample (n=20)')
m_samp, b_samp = np.polyfit(ad_samp, rev_samp, 1)
ax2.plot(x_line, m_samp*x_line + b_samp, color='steelblue', linewidth=2,
         label=f'Fitted line: y={b_samp:.1f}+{m_samp:.1f}x')
ax2.set_xlabel('Advertising spend ($K)')
ax2.set_ylabel('Revenue ($K)')
ax2.set_title('One random sample (n=20)')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()
print(f'Sample estimates: b₀ = {b_samp:.2f},  b₁ = {m_samp:.2f}')

### Variability of the slope from sample to sample

In [ ]:
# Draw 100 different samples → show how b₁ varies
rng12 = np.random.default_rng(470)
fig, ax = plt.subplots()
ax.scatter(ad_pop, rev_pop, alpha=0.2, s=8, color='grey')

slopes = []
for _ in range(100):
    idx_k = rng12.choice(pop_size, size=20, replace=False)
    x_k, y_k = ad_pop[idx_k], rev_pop[idx_k]
    m_k, b_k = np.polyfit(x_k, y_k, 1)
    slopes.append(m_k)
    ax.plot(x_line, m_k*x_line + b_k, color='steelblue', alpha=0.15, linewidth=0.8)

ax.set_xlabel('Advertising spend ($K)')
ax.set_ylabel('Revenue ($K)')
ax.set_title('100 different sample regression lines\n(each from n=20 stores)')
plt.tight_layout()
plt.show()
print(f'True β₁ = {beta1_true}')
print(f'Mean of 100 sample slopes = {np.mean(slopes):.2f}')
print(f'SD of 100 sample slopes   = {np.std(slopes):.2f}  ← this is the SE(b₁)')

### Bootstrap CI for the slope

In [ ]:
rng13 = np.random.default_rng(4748)
n_boot2 = 10_000
boot_slopes = np.zeros(n_boot2)
for i in range(n_boot2):
    idx_bs = rng13.choice(20, size=20, replace=True)
    x_bs, y_bs = ad_samp[idx_bs], rev_samp[idx_bs]
    boot_slopes[i] = np.polyfit(x_bs, y_bs, 1)[0]

ci_slope_95 = np.percentile(boot_slopes, [2.5, 97.5])

fig, ax = plt.subplots()
ax.hist(boot_slopes, bins=50, color='steelblue', edgecolor='white')
ax.axvline(m_samp, color='black', linewidth=2, label=f'Observed b₁ = {m_samp:.2f}')
ax.axvline(ci_slope_95[0], color='red', linestyle='--', label='95% CI')
ax.axvline(ci_slope_95[1], color='red', linestyle='--')
ax.axvline(beta1_true, color='green', linewidth=2, linestyle=':', label=f'True β₁ = {beta1_true}')
ax.set_xlabel('Bootstrap slope estimate (b₁)')
ax.set_ylabel('Count')
ax.set_title('Bootstrap distribution of slope')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Observed b₁ = {m_samp:.2f}')
print(f'95% Bootstrap CI: ({ci_slope_95[0]:.2f}, {ci_slope_95[1]:.2f})')
print(f'The interval captures the true β₁ = {beta1_true}.')

### t-test for the slope

$$H_0: \beta_1 = 0 \quad H_A: \beta_1 \neq 0$$

$$T = \frac{b_1 - 0}{SE_{b_1}}, \quad df = n - 2$$

In [ ]:
# Fit with statsmodels to get full inference output
sandwich_df = pd.DataFrame({'ad': ad_samp, 'rev': rev_samp})
slr_model = smf.ols('rev ~ ad', data=sandwich_df).fit()
print(slr_model.summary())

In [ ]:
# Extract key values
b0_hat = slr_model.params['Intercept']
b1_hat = slr_model.params['ad']
se_b1  = slr_model.bse['ad']
T_b1   = slr_model.tvalues['ad']
p_b1   = slr_model.pvalues['ad']
r2     = slr_model.rsquared

print(f'Fitted model: revenue = {b0_hat:.2f} + {b1_hat:.2f} × advertising')
print(f'SE(b₁) = {se_b1:.2f}')
print(f'T = {b1_hat:.2f} / {se_b1:.2f} = {T_b1:.2f}')
print(f'p-value = {p_b1:.4f}')
print(f'R² = {r2:.3f}')
print()
ci_b1 = slr_model.conf_int().loc['ad']
print(f'95% CI for β₁: ({ci_b1[0]:.2f}, {ci_b1[1]:.2f})')

---
## 8.2 Checking Model Assumptions: L-I-N-E

Before trusting inference from a regression model, verify four conditions — remembered with the acronym **L-I-N-E**:

| Letter | Condition | Diagnostic |
|--------|-----------|------------|
| **L** | **Linear** relationship between x and y | Residuals vs. fitted (no pattern) |
| **I** | **Independent** observations | Study design |
| **N** | **Normal** residuals | Q-Q plot of residuals |
| **E** | **Equal** variance (homoscedasticity) | Residuals vs. fitted (constant spread) |

In [ ]:
fitted = slr_model.fittedvalues
residuals = slr_model.resid

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# L & E: Residuals vs. Fitted
ax = axes[0]
ax.scatter(fitted, residuals, color='steelblue', alpha=0.7)
ax.axhline(0, color='red', linestyle='--')
ax.set_xlabel('Fitted values')
ax.set_ylabel('Residuals')
ax.set_title('Residuals vs. Fitted\n(check L and E)')

# N: Normal Q-Q plot
ax2 = axes[1]
(osm, osr), (slope, intercept, r) = stats.probplot(residuals, dist='norm')
ax2.scatter(osm, osr, color='steelblue', alpha=0.7)
ax2.plot(osm, slope*np.array(osm) + intercept, color='red', linestyle='--')
ax2.set_xlabel('Theoretical quantiles')
ax2.set_ylabel('Sample quantiles')
ax2.set_title('Normal Q-Q plot\n(check N)')

# E: Scale-location (sqrt of abs residuals vs fitted)
ax3 = axes[2]
ax3.scatter(fitted, np.sqrt(np.abs(residuals)), color='steelblue', alpha=0.7)
ax3.set_xlabel('Fitted values')
ax3.set_ylabel('√|Residuals|')
ax3.set_title('Scale-Location\n(check E — equal variance)')

plt.tight_layout()
plt.show()

print('Interpretation:')
print('L: Residuals vs. fitted should show NO pattern → linear relationship is ok')
print('N: Q-Q plot points should fall near the diagonal → residuals are approx. normal')
print('E: Scale-location should show constant spread → equal variance (homoscedasticity)')

### Example of a VIOLATED assumption: non-linear relationship

In [ ]:
rng14 = np.random.default_rng(99)
x_nl = np.linspace(1, 10, 50)
y_nl = 2 * x_nl**2 - 3*x_nl + rng14.normal(0, 5, 50)   # truly quadratic

bad_model = smf.ols('y ~ x', data=pd.DataFrame({'x': x_nl, 'y': y_nl})).fit()
res_bad = bad_model.resid
fit_bad = bad_model.fittedvalues

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.scatter(x_nl, y_nl, color='steelblue')
ax.plot(x_nl, bad_model.params[0] + bad_model.params[1]*x_nl, color='red')
ax.set_title('Data with quadratic trend, linear fit')
ax.set_xlabel('x'); ax.set_ylabel('y')

ax2 = axes[1]
ax2.scatter(fit_bad, res_bad, color='steelblue')
ax2.axhline(0, color='red', linestyle='--')
ax2.set_title('Residuals vs. Fitted — CURVED pattern\n→ Linearity (L) is VIOLATED')
ax2.set_xlabel('Fitted values'); ax2.set_ylabel('Residuals')

plt.tight_layout()
plt.show()

---
## 8.3 Inference for Multiple Linear Regression

With multiple predictors, each coefficient gets its own t-test:

$$H_0: \beta_j = 0 \quad (\text{predictor } j \text{ has no effect, holding others fixed})$$

### Example: Predicting loan interest rates

Using the `loans_full_schema` dataset, we predict **interest rate** from three predictors:
- `debt_to_income` — ratio of debt to income
- `term` — loan term (36 or 60 months)
- `credit_checks` — inquiries in the last 12 months

In [ ]:
# Place loans_full_schema.csv in this folder before running
loans_raw = pd.read_csv('loans_full_schema.csv')

# Prepare variables to match the book
loans = loans_raw[['interest_rate', 'debt_to_income', 'term', 'inquiries_last_12m']].copy()
loans = loans.rename(columns={'inquiries_last_12m': 'credit_checks'})
loans = loans.dropna()

print(f'Rows after dropping NAs: {len(loans)}')
print(loans[['interest_rate','debt_to_income','term','credit_checks']].describe().round(2))

In [ ]:
mlr_model = smf.ols(
    'interest_rate ~ debt_to_income + term + credit_checks',
    data=loans
).fit()

print(mlr_model.summary())

In [ ]:
# Display a clean coefficient table
coef_table = pd.DataFrame({
    'Estimate': mlr_model.params,
    'Std. Error': mlr_model.bse,
    't value': mlr_model.tvalues,
    'p-value': mlr_model.pvalues
}).round(4)
print('Regression coefficients:')
print(coef_table)
print(f'\nR² = {mlr_model.rsquared:.4f}')
print(f'Adjusted R² = {mlr_model.rsquared_adj:.4f}')
print()
print('Interpretation of coefficients:')
print(f'  Intercept:      Baseline interest rate = {mlr_model.params[0]:.2f}%')
print(f'  debt_to_income: Each 1-unit increase raises rate by {mlr_model.params[1]:.3f}%')
print(f'  term:           60-mo vs 36-mo loans: +{mlr_model.params[2]:.2f}% rate')
print(f'  credit_checks:  Each additional inquiry raises rate by {mlr_model.params[3]:.2f}%')

In [ ]:
# Diagnostic plots for MLR
fitted_mlr = mlr_model.fittedvalues
resid_mlr  = mlr_model.resid

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].scatter(fitted_mlr, resid_mlr, alpha=0.2, s=5, color='steelblue')
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel('Fitted values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs. Fitted')

(osm2, osr2), (sl2, int2, _) = stats.probplot(resid_mlr, dist='norm')
axes[1].scatter(osm2, osr2, alpha=0.3, s=5, color='steelblue')
axes[1].plot(osm2, sl2*np.array(osm2)+int2, 'r--')
axes[1].set_xlabel('Theoretical quantiles')
axes[1].set_ylabel('Sample quantiles')
axes[1].set_title('Normal Q-Q plot')

axes[2].hist(resid_mlr, bins=50, color='steelblue', edgecolor='white')
axes[2].set_xlabel('Residuals')
axes[2].set_title('Histogram of residuals')

plt.tight_layout()
plt.show()

### R² vs. Adjusted R²

Adding more predictors always increases $R^2$, even if they are meaningless.
**Adjusted $R^2$** penalizes for the number of predictors:

$$R^2_{adj} = 1 - \frac{(1-R^2)(n-1)}{n-p-1}$$

Use $R^2_{adj}$ for model comparison — prefer the model with the **higher** $R^2_{adj}$.

In [ ]:
# Compare models with different numbers of predictors
model_1 = smf.ols('interest_rate ~ debt_to_income', data=loans).fit()
model_2 = smf.ols('interest_rate ~ debt_to_income + term', data=loans).fit()
model_3 = smf.ols('interest_rate ~ debt_to_income + term + credit_checks', data=loans).fit()

comparison = pd.DataFrame({
    'Predictors': ['debt_to_income', '+ term', '+ credit_checks'],
    'R²':         [model_1.rsquared, model_2.rsquared, model_3.rsquared],
    'Adj. R²':    [model_1.rsquared_adj, model_2.rsquared_adj, model_3.rsquared_adj],
    'AIC':        [model_1.aic, model_2.aic, model_3.aic]
}).round(4)

print(comparison.to_string(index=False))
print()
print('Adding each predictor increases R² AND adjusted R² → each is useful.')
print('Prefer model 3 by Adj. R² and AIC (lower AIC = better fit/parsimony).')

---
## 8.4 Inference for Logistic Regression

When the response is **binary** (0/1), we use logistic regression:

$$\log\left(\frac{p}{1-p}\right) = \beta_0 + \beta_1 x_1 + \cdots + \beta_k x_k$$

Coefficients are interpreted as **log-odds** changes. Exponentiated coefficients are **odds ratios**.

**Conditions:**
1. Each $Y_i$ is independent.
2. Each predictor $x_i$ is linearly related to $\text{logit}(p_i)$.

### Example: Email spam classification

Can we predict whether an email is **spam** based on its characteristics?

In [ ]:
# Place email.csv in this folder before running
email = pd.read_csv('email.csv')
print(f'Dataset: {email.shape[0]} emails, {email.shape[1]} variables')
print(f'Spam rate: {email["spam"].mean():.1%}')
print(email[['spam','to_multiple','attach','dollar','winner',
             'format','re_subj','exclaim_mess']].head())

In [ ]:
# Prepare the data
email_clean = email[['spam','to_multiple','attach','dollar',
                      'format','re_subj','exclaim_mess']].dropna().copy()
email_clean['winner'] = (email['winner'] == 'yes').astype(int)

# Fit logistic regression
logit_model = smf.logit(
    'spam ~ to_multiple + attach + dollar + winner + format + re_subj + exclaim_mess',
    data=email_clean
).fit()

print(logit_model.summary())

In [ ]:
# Coefficient table with odds ratios
coef_df = pd.DataFrame({
    'Log-Odds': logit_model.params,
    'Odds Ratio': np.exp(logit_model.params),
    'p-value': logit_model.pvalues
}).round(4)

print('Logistic Regression Coefficients:')
print(coef_df)
print()
print('Interpretation examples:')

for var in ['to_multiple', 'winner', 're_subj']:
    coef = logit_model.params.get(var, None)
    if coef is not None:
        direction = 'increases' if coef > 0 else 'decreases'
        print(f'  {var}: log-odds {direction} by {abs(coef):.3f} '
              f'(OR = {np.exp(coef):.3f})')

In [ ]:
# Visualize: fitted probabilities by true class
pred_probs = logit_model.predict(email_clean)

fig, ax = plt.subplots(figsize=(9, 4))
for label, color in [(0, 'steelblue'), (1, 'tomato')]:
    mask = email_clean['spam'] == label
    ax.scatter(
        pred_probs[mask],
        np.zeros(mask.sum()) + label + np.random.uniform(-0.2, 0.2, mask.sum()),
        alpha=0.15, s=8, color=color
    )
ax.set_xlabel('Predicted probability of spam')
ax.set_yticks([0, 1])
ax.set_yticklabels(['Not Spam (0)', 'Spam (1)'])
ax.set_title('Predicted spam probabilities by true label')
ax.axvline(0.5, color='black', linestyle='--', label='Decision boundary (p=0.5)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Model performance: confusion matrix at threshold 0.5
y_pred = (pred_probs >= 0.5).astype(int)
y_true = email_clean['spam']

from sklearn.metrics import confusion_matrix, classification_report
cm = confusion_matrix(y_true, y_pred)
print('Confusion Matrix (threshold = 0.5):')
print(pd.DataFrame(cm,
    index=['Actual: Not Spam', 'Actual: Spam'],
    columns=['Pred: Not Spam', 'Pred: Spam']))
print()
print(classification_report(y_true, y_pred, target_names=['Not Spam', 'Spam']))

### Three scales of logistic regression

Logistic regression can be interpreted on three scales:

| Scale | Formula | Interpretation |
|-------|---------|----------------|
| **Log-odds** | $\beta_0 + \beta_1 x$ | Additive, linear |
| **Odds** | $e^{\beta_0 + \beta_1 x}$ | Multiplicative (odds ratio) |
| **Probability** | $\frac{1}{1+e^{-(\beta_0+\beta_1 x)}}$ | Intuitive but nonlinear |

In [ ]:
# Visualize the three scales using exclaim_mess as the predictor
simple_logit = smf.logit('spam ~ exclaim_mess', data=email_clean).fit(disp=0)

b0 = simple_logit.params['Intercept']
b1 = simple_logit.params['exclaim_mess']

x_vals = np.linspace(0, email_clean['exclaim_mess'].quantile(0.99), 200)
log_odds = b0 + b1 * x_vals
odds = np.exp(log_odds)
probs = 1 / (1 + np.exp(-log_odds))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
titles = ['Log-Odds scale', 'Odds scale', 'Probability scale']
ys = [log_odds, odds, probs]
ylabels = ['Log-odds of spam', 'Odds of spam', 'P(spam)']

for ax, title, y, ylabel in zip(axes, titles, ys, ylabels):
    ax.plot(x_vals, y, color='steelblue', linewidth=2)
    ax.set_xlabel('Number of exclamation marks')
    ax.set_ylabel(ylabel)
    ax.set_title(title)

plt.suptitle('Logistic regression for spam ~ exclaim_mess\n(three interpretive scales)',
             y=1.02)
plt.tight_layout()
plt.show()

print(f'b₁ = {b1:.4f} (log-odds scale)')
print(f'OR = e^b₁ = {np.exp(b1):.4f}')
print('Each additional exclamation mark multiplies the odds of spam by '
      f'{np.exp(b1):.4f}.')

---
## Summary

### Chapter 7 — Inference for Numerical Data

| Setting | Point estimate | Test statistic | df |
|---------|---------------|----------------|----|
| One mean | $\bar{x}$ | $T = (\bar{x} - \mu_0) / (s/\sqrt{n})$ | $n-1$ |
| Paired means | $\bar{x}_{diff}$ | $T = \bar{x}_{diff} / (s_{diff}/\sqrt{n})$ | $n-1$ |
| Two means | $\bar{x}_1 - \bar{x}_2$ | $T = (\bar{x}_1 - \bar{x}_2) / SE$ | Welch approx |
| Many means | — | $F = MSG / MSE$ | $(k-1, n-k)$ |

### Chapter 8 — Inference for Regression

| Model | Null hypothesis | Test statistic | df |
|-------|----------------|----------------|----|
| SLR slope | $\beta_1 = 0$ | $T = b_1 / SE_{b_1}$ | $n-2$ |
| MLR coefficient | $\beta_j = 0$ | $T = b_j / SE_{b_j}$ | $n-p-1$ |
| Logistic coefficient | $\beta_j = 0$ | $z = b_j / SE_{b_j}$ | Large-sample normal |

**L-I-N-E** conditions for regression:
- **L**inear: no pattern in residuals vs. fitted
- **I**ndependent: study design
- **N**ormal: residuals follow a normal distribution (Q-Q plot)
- **E**qual variance: constant spread in residuals vs. fitted